:meth:`SeqMut.evolve` is the ML-guided directed-evolution loop: each round scans the current sequence, fixes the substitution with the best ``delta_pred`` into the background, and repeats for ``depth`` rounds — stacking an interpretable multi-mutation variant without a combinatorial search.

In [1]:
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import aaanalysis as aa
aa.options["verbose"] = False

# Data, CPP features, and a fitted TreeModel that scores each sequence
df_seq = aa.load_dataset(name="DOM_GSEC", n=10)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
split_kws = sf.get_split_kws()
df_scales = aa.load_scales()
cpp = aa.CPP(df_parts=df_parts, split_kws=split_kws, df_scales=df_scales, verbose=False)
df_feat = cpp.run(labels=labels, n_filter=25)
X = sf.feature_matrix(features=list(df_feat["feature"]), df_parts=df_parts, df_scales=df_scales)
tm = aa.TreeModel().fit(X, labels=labels)
entry = df_seq["entry"].iloc[0]
ts = int(df_seq.set_index("entry").loc[entry, "tmd_start"])

seqmut = aa.SeqMut(model=tm)
df_evolve = seqmut.evolve(df_seq=df_seq.head(2), df_feat=df_feat, depth=3, region="tmd")
aa.display_df(df_evolve, n_rows=10, show_shape=True)

/Users/stephanbreimann/Programming/1Packages/wt-seqmut-ml-guided/aaanalysis/feature_engineering/_backend/cpp_run.py:163: UserWarning: CPP is using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.
  warnings.warn(


DataFrame shape: (6, 7)


,entry,round,mutation,sequence_mut,delta_cpp,shift_score,delta_pred
1,Q14802,1,G52A,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,3.415660,3.415660,35.300000
2,Q14802,2,V47L,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,4.291570,4.291570,56.300000
3,Q14802,3,I54D,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,4.578230,4.578230,61.900000
4,Q86UE4,1,Y69L,MAARSWQDELAQQAE...SPKQIKKKKKARRET,1.720250,1.720250,33.000000
5,Q86UE4,2,V54E,MAARSWQDELAQQAE...SPKQIKKKKKARRET,2.850090,2.850090,48.200000
6,Q86UE4,3,W71P,MAARSWQDELAQQAE...SPKQIKKKKKARRET,3.303090,3.303090,62.100000


``region`` restricts each round's scan (a part name or explicit positions) and ``to_aa`` restricts the substitution alphabet; ``jmd_n_len`` / ``jmd_c_len`` set the flanking lengths.

In [2]:
df_evolve2 = seqmut.evolve(df_seq=df_seq.head(2), df_feat=df_feat, depth=2,
                          region="tmd", to_aa=["A", "L", "G"], jmd_n_len=10, jmd_c_len=10)
aa.display_df(df_evolve2, n_rows=10, show_shape=True)

DataFrame shape: (4, 7)


,entry,round,mutation,sequence_mut,delta_cpp,shift_score,delta_pred
1,Q14802,1,G52A,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,3.415660,3.415660,35.300000
2,Q14802,2,V47L,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,4.291570,4.291570,56.300000
3,Q86UE4,1,Y69L,MAARSWQDELAQQAE...SPKQIKKKKKARRET,1.720250,1.720250,33.000000
4,Q86UE4,2,V54A,MAARSWQDELAQQAE...SPKQIKKKKKARRET,2.654590,2.654590,47.800000
